In [0]:
%pip install pymupdf

In [0]:
%pip install pdfplumber azure-identity azure-storage-file-datalake azure-storage-blob

In [0]:
%restart_python

In [0]:
def blob_source(output_format, page, file_name):
    try:
        import pandas as pd
        from io import BytesIO
        from azure.storage.blob import BlobServiceClient
        from datetime import date
        from pyspark.sql.functions import split,regexp_extract
        import pdfplumber

        sas_token = dbutils.secrets.get("pdf-sas", "blob-sas-tokn")
        blob_url = "https://pdfexcelacc.blob.core.windows.net/"
        container_name = "source"

        blob_service_client = BlobServiceClient(account_url=blob_url, credential=sas_token)
        container_client = blob_service_client.get_container_client(container_name)
        for blob in container_client.list_blobs():
            blob_client = container_client.get_blob_client(blob)
            blob_data = blob_client.download_blob().readall()
            pdf_stream = BytesIO(blob_data)
            data = []
            with pdfplumber.open(pdf_stream) as pdf:
                for p in pdf.pages:
                    table = p.extract_table()
                    if table:
                        data.extend(table)
            if not data:
                raise ValueError("No table data found in PDF.")
            header = data[0]
            rows = data[1:]
            pdf_df = pd.DataFrame(rows, columns=header)
            pdf_df = pdf_df.dropna(how='all')
            pdf_df.columns = [c.strip().lower().replace(" ", ",") for c in pdf_df.columns]
            full_string = 'tailnum,type,manufacturer,issue_date,model,status,aircraft_type,engine_type,year'
            spark_df = spark.createDataFrame(pdf_df)
            new_df = spark_df.select(
                    regexp_extract(full_string, r'^(\S+)', 1).alias('tailnum'),
                    regexp_extract(full_string, r'^\S+\s+(\S+)', 1).alias('type'),
                    regexp_extract(full_string, r'^\S+\s+\S+\s+(.+?)\s+\d{4}-\d{2}-\d{2}', 1).alias('manufacturer'),
                    regexp_extract(full_string, r'(\d{4}-\d{2}-\d{2})', 1).alias('issue_date'),
                    regexp_extract(full_string, r'\d{4}-\d{2}-\d{2}(\S+)',1).alias('model'),
                    regexp_extract(full_string, r'(Valid|Invalid)',1).alias('status'),
                    regexp_extract(full_string, r'(Fixed Wing)',1).alias('aircraft_type'),
                    regexp_extract(full_string, r'(Multi-Engine Turbo-Fan | Multi-Engine Turbo-Jet)',1).alias('engine_type'),
                    regexp_extract(full_string, r'(\S+)$',1).alias('year')
                )
            output_path = (
                f'abfss://raw@datalakestrgdev.dfs.core.windows.net/'
                f'{file_name.split(".")[0]}/Datepart={date.today()}'
            )
            # spark_df = spark.createDataFrame(new_df)
            new_df.coalesce(1).write.mode("overwrite").option("header", "true").format("csv").save(output_path)
            files = dbutils.fs.ls(output_path)
            for f in files:
                if f.name.startswith("part-") and f.name.endswith(".csv"):
                    dest_path = (
                        f"abfss://raw@datalakestrgdev.dfs.core.windows.net/"
                        f"{file_name.split('.')[0]}/Datepart={date.today()}/"
                        f"{file_name.split('.')[0]}.csv"
                    )
                    dbutils.fs.mv(f.path, dest_path)
                    dbutils.fs.rm(
                        f"abfss://raw@datalakestrgdev.dfs.core.windows.net/"
                        f"{file_name.split('.')[0]}/Datepart={date.today()}/_SUCCESS",
                        True
                    )
            # convert_into_table(f'abfss://raw@datalakdatalakestrgdevsink1.dfs.core.windows.net/{file_name.split(".")[0]}/Datepart={date.today()}/{file_name.split('.')[0]}.csv')
    except Exception as e:
        print(f"Error: {e}")

In [0]:
list_files = [(i.name,i.name.split(".")[1]) for i in dbutils.fs.ls("abfss://source@pdfexcelacc.dfs.core.windows.net/") if i.name.split(".")[1] == "pdf" ]
from pyspark.sql.functions import *
from datetime import date

for i in list_files:
    print(i)
    blob_source("csv","all",i[0])
    # convert_into_table(f'abfss://raw@datalakesink1.dfs.core.windows.net/{i[0].split('.')[0]}/Datepart={date.today()}/{i[0].split('.')[0]}.csv')